In [1]:
#Modify the path to a directory on your machine
import os
os.environ["CRDS_PATH"] = "/home/hailin/Documents/CRDS"
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"

# Packages that allow us to get information about objects:
import asdf
import copy
import shutil

# Numpy library:
import numpy as np

# For downloading data
import requests

# Astropy tools:
from astropy.io import fits
from astropy.utils.data import download_file
from astropy.visualization import ImageNormalize, ManualInterval, LogStretch

import matplotlib.pyplot as plt
import matplotlib as mpl

# Plotting tools:
from pipeline1_plotting_tools import download_files, plot_jump, plot_jumps, plot_ramp, plot_ramps, show_image, side_by_side

# Use this version for non-interactive plots (easier scrolling of the notebook)
%matplotlib inline

# Use this version (outside of Jupyter Lab) if you want interactive plots
#%matplotlib notebook

# List of possible data quality flags
from jwst.datamodels import dqflags

# The entire calwebb_detector1 pipeline
from jwst.pipeline import calwebb_detector1

# Individual steps that make up calwebb_detector1
from jwst.dq_init import DQInitStep
from jwst.saturation import SaturationStep
from jwst.superbias import SuperBiasStep
from jwst.ipc import IPCStep                                                                                    
from jwst.refpix import RefPixStep                                                                
from jwst.linearity import LinearityStep
from jwst.persistence import PersistenceStep
from jwst.dark_current import DarkCurrentStep
from jwst.jump import JumpStep
from jwst.ramp_fitting import RampFitStep
from jwst import datamodels

import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from pathlib import Path

import jwst
print(jwst.__version__)

1.15.1


In [2]:
data_path = Path('../data/JWST')
for item in data_path.iterdir():
    input_file_base = item.name

    jump_file = '../data/JWST/' + input_file_base + '/' + input_file_base + '_jumpstep.fits'
    jump = datamodels.open(jump_file)
    photo_start = 0
    photo_end   = 244
    # bin 的范围
    dn_min = -200
    dn_max = 400
    # Create a 4-dimensional map of ANY flags
    dq_map_2d = np.sum(jump.groupdq[0, :, :, :] > 0, axis=0)
    # Determine how many pixels are not GOOD
    dq_map_indexes = np.where(dq_map_2d > 0)
    bad_pix = np.sum(dq_map_2d > 0)
    total_pix = 2048 * 2048
    pre_mask = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]


    # 现在mask和原bad pixel mask合并
    raw_photo_3 = pre_mask
    raw_photo_3[dq_map_indexes] = np.nan
    raw_photo_flatten_3 = raw_photo_3.flatten()
    nan_mask_3 = np.isnan(raw_photo_flatten_3)
    photo_masked = raw_photo_flatten_3[~nan_mask_3]
    masked_counts, masked_bin_edges = np.histogram(photo_masked, bins=range(dn_min, dn_max+1))

    # 保存结果
    result = np.array([range(dn_min,dn_max), masked_counts]).T
    np.savetxt('./results/unmasked'+ input_file_base +'.txt',result)
    print(input_file_base + ' analysis completed')

jw01121144001_02102_00001_nrs2 analysis completed
jw01121156001_02102_00001_nrs2 analysis completed
jw01121108001_02102_00002_nrs2 analysis completed
jw01121138001_02102_00001_nrs2 analysis completed
jw01121150001_02102_00001_nrs2 analysis completed
jw01121146001_02102_00002_nrs2 analysis completed
jw01121108001_02102_00001_nrs2 analysis completed
jw01121142001_02102_00001_nrs2 analysis completed
jw01121160001_02102_00001_nrs2 analysis completed
jw01121152001_02102_00002_nrs2 analysis completed
jw01121114001_02102_00002_nrs2 analysis completed
jw01121138001_02102_00002_nrs2 analysis completed
jw01121150001_02102_00002_nrs2 analysis completed
jw01121152001_02102_00001_nrs2 analysis completed
jw01121144001_02102_00002_nrs2 analysis completed
jw01121160001_02102_00002_nrs2 analysis completed
jw01121146001_02102_00001_nrs2 analysis completed
jw01121114001_02102_00001_nrs2 analysis completed
jw01121142001_02102_00002_nrs2 analysis completed
jw01121156001_02102_00002_nrs2 analysis completed
